# 03 — Analysis Corpus: Language ID, De-duplication, Boilerplate

Input: the clean AI corpus from 02D (`clean_ai_docs.parquet`, `clean_ai_blocks.parquet`).

Three corrections before any modeling (`pipeline/s10_corpus.py`):

- **Language ID** with fastText `lid.176`. The source `language` field is `en` for every row, including Portuguese, Galician and Kazakh pages. A document is kept when the top language is English with probability ≥ 0.70, so low-confidence pages are removed along with clearly non-English ones.
- **Near-duplicate documents** with MinHash LSH (word 5-shingles, 128 permutations, Jaccard ≥ 0.8). Syndicated wire stories and templated pages form clusters; the earliest copy is kept, so one story counts once.
- **Boilerplate blocks**: a paragraph whose normalized text appears in 20+ analysis documents is flagged and dropped; exact repeats below that threshold keep one copy.

Press-release sources are flagged (`is_press_release`) rather than dropped, so later results can be checked with and without them.

Outputs in `results/corpus/`: `docs.parquet`, `blocks.parquet`, `corpus_stats.json`.

In [1]:
import os, sys

PROJECT_DIR = "/content/drive/MyDrive/NLP/NLP_FINAL_PROJECT_Tom_Chen"
IN_COLAB = os.path.isdir("/content")
if IN_COLAB and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.environ.setdefault("PROJECT_DIR", PROJECT_DIR)
# Path to a checkout of this repository (on Drive in Colab).
sys.path.insert(0, os.environ.get("REPO_DIR", f"{os.environ['PROJECT_DIR']}/code"))

import json
import pandas as pd
from pipeline import config

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [2]:
RUN = False  # outputs are on Drive; set True to rebuild (CPU, ~5 min)
if RUN:
    from pipeline import s10_corpus
    s10_corpus.main()

## Corpus funnel

In [3]:
stats = json.load(open(config.out("corpus", "corpus_stats.json")))
pd.Series({k: v for k, v in stats.items() if not isinstance(v, (dict, list))}).to_frame("value")

,value
input_docs,181008
input_blocks,731989
non_english_docs,779
near_dup_clusters_multi,11661
non_canonical_docs,49941
analysis_docs,124741
analysis_blocks,400662
boilerplate_blocks,12125
press_release_share_analysis,0.105699
date_min,2022-01-01


## Largest near-duplicate clusters

The biggest clusters are templated pages (an AI art gallery site) and AP-style stories syndicated across local TV and newspaper sites.

In [4]:
pd.DataFrame(stats["largest_clusters"]).head(12)

,n,title
0,168,"Native Stew - Bahamas AI Art, Photos, Videos: Seagull Bay"
1,156,"Native Stew - Bahamas AI Art, Photos, Videos: Imagination is Power"
2,99,"Native Stew - Bahamas AI Art, Photos, Videos: Pleasure Boating in The Bahamas: A Comprehensive Guide"
3,80,Trump arrested? Putin jailed? Fake AI images spread online
4,61,Is Bing too belligerent? Microsoft looks to tame AI chatbot
5,59,The threat of wildfires is rising. So are new artificial intelligence solutions to fight them | KLFY.com
6,53,"John Grisham, George R.R. Martin and more authors sue OpenAI for copyright infringement"
7,52,"Paper exams, chatbot bans: Colleges seek to ‘ChatGPT-proof’ assignments"
8,51,Lawyers blame ChatGPT for tricking them into citing bogus case law | KOIN.com
9,48,"Apple doubles down on artificial intelligence, announcing partnership with OpenAI"


In [5]:
docs = pd.read_parquet(config.out("corpus", "docs.parquet"),
                       columns=["doc_id", "date", "domain", "title", "lang", "dup_cluster", "cluster_size",
                                "is_canonical", "is_press_release", "in_analysis"])
sizes = docs.groupby("dup_cluster").size()
print(f"clusters with >1 copy: {(sizes > 1).sum():,}; documents removed as copies: {(~docs.is_canonical).sum():,}")
sizes.value_counts().sort_index().head(10).rename("n_clusters").to_frame()

clusters with >1 copy: 11,661; documents removed as copies: 49,941


,n_clusters
1,119406
2,5271
3,1438
4,781
5,581
6,516
7,459
8,420
9,349
10,328


Example: one syndicated story and its copies.

In [6]:
example = sizes[sizes.between(4, 6)].index[3]
docs[docs.dup_cluster == example][["date", "domain", "title", "is_canonical"]]

,date,domain,title,is_canonical
74238,2023-11-16,kfyrtv.com,'Do It Right' with AI: Ally creators experiment with generative AI in marketing test case,True
74301,2023-11-16,wctv.tv,'Do It Right' with AI: Ally creators experiment with generative AI in marketing test case,False
74345,2023-11-16,kbtx.com,'Do It Right' with AI: Ally creators experiment with generative AI in marketing test case,False
74443,2023-11-16,kalb.com,'Do It Right' with AI: Ally creators experiment with generative AI in marketing test case,False
74548,2023-11-16,nbc12.com,'Do It Right' with AI: Ally creators experiment with generative AI in marketing test case,False


## Languages removed and source mix

In [7]:
print(pd.Series(stats["lang_counts_top"]).to_string())
ana = docs[docs.in_analysis]
print(f"\npress-release share of the analysis set: {ana.is_press_release.mean():.1%}")
ana.domain.value_counts().head(15).rename("articles").to_frame()

en    180899
fr        29
it        20
es        18
de        13
pt         5
nl         4
sk         4
tl         3
zh         2

press-release share of the analysis set: 10.6%


,articles
domain,
menafn.com,3549
einpresswire.com,3523
nasdaq.com,2316
prnewswire.com,2218
mexc.co,1997
livemint.com,1702
economictimes.indiatimes.com,1612
business-standard.com,1510
globenewswire.com,1247
